In [3]:
!gdown 16mZm3C7xKpPqp_86e64uzduLpM5mPUdq

Downloading...
From: https://drive.google.com/uc?id=16mZm3C7xKpPqp_86e64uzduLpM5mPUdq
To: /Users/shudhanshu/Desktop/Study Projects/GenAI/Agent Implementation/using langchain/comicdb_create_script.sql
100%|██████████████████████████████████████| 14.5k/14.5k [00:00<00:00, 13.9MB/s]


In [4]:
!sqlite3 --version

3.43.2 2023-10-10 13:08:14 1b37c146ee9ebb7acd0160c0ab1fd11017a419fa8a3187386ed8cb32b709aapl (64-bit)


In [5]:
!sqlite3 ComicStore.db ".read ./comicdb_create_script.sql"

In [6]:
!sqlite3 ComicStore.db "SELECT name FROM sqlite_master WHERE type='table';"

Branch
Employee
Publisher
Comic
Inventory
Customer
Sale
SaleTransactions


In [7]:
from langchain_community.utilities import SQLDatabase

In [9]:
db = SQLDatabase.from_uri("sqlite:///ComicStore.db")
db

In [10]:
print(db.dialect)
print(db.get_usable_table_names())

sqlite
['Branch', 'Comic', 'Customer', 'Employee', 'Inventory', 'Publisher', 'Sale', 'SaleTransactions']


In [11]:
db.run("SELECT * FROM Comic LIMIT 10", include_columns=True)

"[{'ComicId': 1, 'Title': 'Spider-Man: Homecoming', 'PublisherId': 1, 'Genre': 'Superhero', 'Price': 19.99, 'ReleaseDate': '2017-07-07'}, {'ComicId': 2, 'Title': 'Batman: Year One', 'PublisherId': 2, 'Genre': 'Superhero', 'Price': 14.99, 'ReleaseDate': '1987-02-01'}, {'ComicId': 3, 'Title': 'Hellboy: Seed of Destruction', 'PublisherId': 3, 'Genre': 'Supernatural', 'Price': 24.99, 'ReleaseDate': '1994-10-01'}, {'ComicId': 4, 'Title': 'Saga Volume 1', 'PublisherId': 4, 'Genre': 'Fantasy', 'Price': 12.99, 'ReleaseDate': '2012-03-14'}, {'ComicId': 5, 'Title': 'Transformers: All Hail Megatron', 'PublisherId': 5, 'Genre': 'Science Fiction', 'Price': 25.99, 'ReleaseDate': '2008-09-01'}, {'ComicId': 6, 'Title': 'X-Men: Days of Future Past', 'PublisherId': 1, 'Genre': 'Superhero', 'Price': 18.99, 'ReleaseDate': '1981-01-01'}, {'ComicId': 7, 'Title': 'The Killing Joke', 'PublisherId': 2, 'Genre': 'Superhero', 'Price': 14.99, 'ReleaseDate': '1988-03-29'}, {'ComicId': 8, 'Title': 'Sin City: The Ha

In [13]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('ComicStore.db')
pd.read_sql_query("SELECT * FROM Comic LIMIT 10", conn)

,ComicId,Title,PublisherId,Genre,Price,ReleaseDate
0,1,Spider-Man: Homecoming,1,Superhero,19.99,2017-07-07
1,2,Batman: Year One,2,Superhero,14.99,1987-02-01
2,3,Hellboy: Seed of Destruction,3,Supernatural,24.99,1994-10-01
3,4,Saga Volume 1,4,Fantasy,12.99,2012-03-14
4,5,Transformers: All Hail Megatron,5,Science Fiction,25.99,2008-09-01
5,6,X-Men: Days of Future Past,1,Superhero,18.99,1981-01-01
6,7,The Killing Joke,2,Superhero,14.99,1988-03-29
7,8,Sin City: The Hard Goodbye,4,Noir,22.99,1991-06-01
8,9,Usagi Yojimbo Volume 1,5,Adventure,20.99,1987-09-01
9,10,Deadpool: Merc with a Mouth,1,Superhero,16.99,2009-08-05


In [15]:
print(db.get_table_info(table_names=['Comic', 'Sale']))


CREATE TABLE "Comic" (
	"ComicId" INTEGER NOT NULL, 
	"Title" NVARCHAR(100) NOT NULL, 
	"PublisherId" INTEGER NOT NULL, 
	"Genre" NVARCHAR(50), 
	"Price" NUMERIC(10, 2) NOT NULL, 
	"ReleaseDate" DATETIME, 
	PRIMARY KEY ("ComicId"), 
	FOREIGN KEY("PublisherId") REFERENCES "Publisher" ("PublisherId")
)

/*
3 rows from Comic table:
ComicId	Title	PublisherId	Genre	Price	ReleaseDate
1	Spider-Man: Homecoming	1	Superhero	19.99	2017-07-07 00:00:00
2	Batman: Year One	2	Superhero	14.99	1987-02-01 00:00:00
3	Hellboy: Seed of Destruction	3	Supernatural	24.99	1994-10-01 00:00:00
*/


CREATE TABLE "Sale" (
	"SaleId" INTEGER NOT NULL, 
	"CustomerId" INTEGER NOT NULL, 
	"EmployeeId" INTEGER, 
	"SaleDate" DATETIME NOT NULL, 
	"TotalAmount" NUMERIC(10, 2) NOT NULL, 
	PRIMARY KEY ("SaleId"), 
	FOREIGN KEY("EmployeeId") REFERENCES "Employee" ("EmployeeId"), 
	FOREIGN KEY("CustomerId") REFERENCES "Customer" ("CustomerId")
)

/*
3 rows from Sale table:
SaleId	CustomerId	EmployeeId	SaleDate	TotalAmount
1	1	

In [16]:
from langchain_core.prompts.prompt import PromptTemplate


/Users/shudhanshu/Desktop/Study Projects/GenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [17]:
PROMPT_SUFFIX = """
    Only use the following tables:
    {table_info}

    Question: {input}
"""

_sqlite_prompt = """
    You are a SQLite expert.
    Given an input question, first create a syntactically corrrect SQLite query to run,
    then look at the results of the query and return the answer to the input question.

    Unless the user specifies in the question a specific number of examples to obtain,
    query for at most {top_k} results using the LIMIT clause as per SQLite.

    You canorder the results to return the most informative data in the database.
    Never query for all columns from the table.

    You must query only columns that are needed to answer the question.
    Wrap each column name in double quotes (") to denote them as delimited identifiers.

    Pay attention to use only the column names you can see in the tables below.
    Be careful to not query for columns that do not exist.
    Also, pay attention to which column is in which table.
    Pay attention to use date('now') function to get the current_date, if the question involves "today".
    Pay attention to use table JOINS as necessary if you are adding relevant fields from different tables.

    Generate the output in the exact following formats:

    SQLQuery: SQL query to run
    SQLResult: Result of the SQLQuery
    Answer: Final answer here

    The SQLQuery field above should have the correct SQLite query as plain text without any formatting or code blocks.
    Do not include sql or similar markers.
    Do not try to explain the query, just provide the query as-is, like this: SELECT ...
"""

In [18]:
SQLITE_PROMPT = PromptTemplate(
    input_variables=['input','table_info','top_k'],
    template=_sqlite_prompt + PROMPT_SUFFIX
)

In [19]:
SQLITE_PROMPT

PromptTemplate(input_variables=['input', 'table_info', 'top_k'], input_types={}, partial_variables={}, template='\n    You are a SQLite expert.\n    Given an input question, first create a syntactically corrrect SQLite query to run,\n    then look at the results of the query and return the answer to the input question.\n\n    Unless the user specifies in the question a specific number of examples to obtain,\n    query for at most {top_k} results using the LIMIT clause as per SQLite.\n\n    You canorder the results to return the most informative data in the database.\n    Never query for all columns from the table.\n\n    You must query only columns that are needed to answer the question.\n    Wrap each column name in double quotes (") to denote them as delimited identifiers.\n\n    Pay attention to use only the column names you can see in the tables below.\n    Be careful to not query for columns that do not exist.\n    Also, pay attention to which column is in which table.\n    Pay at

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
# from langchain.chains import create_sql_query_chain
from langchain_classic.chains import create_sql_query_chain
llm_gpt = ChatOpenAI(model='gpt-4o', temperature=0)


In [25]:
text2sql_chain = create_sql_query_chain(
    llm=llm_gpt,
    db=db,
    prompt=SQLITE_PROMPT,
    k=5
)

text2sql_chain

RunnableAssign(mapper={
  input: RunnableLambda(...),
  table_info: RunnableLambda(...)
})
| RunnableLambda(lambda x: {k: v for k, v in x.items() if k not in ('question', 'table_names_to_use')})
| PromptTemplate(input_variables=['input', 'table_info'], input_types={}, partial_variables={'top_k': '5'}, template='\n    You are a SQLite expert.\n    Given an input question, first create a syntactically corrrect SQLite query to run,\n    then look at the results of the query and return the answer to the input question.\n\n    Unless the user specifies in the question a specific number of examples to obtain,\n    query for at most {top_k} results using the LIMIT clause as per SQLite.\n\n    You canorder the results to return the most informative data in the database.\n    Never query for all columns from the table.\n\n    You must query only columns that are needed to answer the question.\n    Wrap each column name in double quotes (") to denote them as delimited identifiers.\n\n    Pay att

In [26]:
text2sql_chain.get_prompts()[0].pretty_print()


    You are a SQLite expert.
    Given an input question, first create a syntactically corrrect SQLite query to run,
    then look at the results of the query and return the answer to the input question.

    Unless the user specifies in the question a specific number of examples to obtain,
    query for at most 5 results using the LIMIT clause as per SQLite.

    You canorder the results to return the most informative data in the database.
    Never query for all columns from the table.

    You must query only columns that are needed to answer the question.
    Wrap each column name in double quotes (") to denote them as delimited identifiers.

    Pay attention to use only the column names you can see in the tables below.
    Be careful to not query for columns that do not exist.
    Also, pay attention to which column is in which table.
    Pay attention to use date('now') function to get the current_date, if the question involves "today".
    Pay attention to use table JOINS as n

In [27]:
response = text2sql_chain.invoke({"question": "Top 5 most popular comics"})
print(response)

SELECT "Comic"."Title", SUM("SaleTransactions"."Quantity") AS "TotalSold"
FROM "SaleTransactions"
JOIN "Comic" ON "SaleTransactions"."ComicId" = "Comic"."ComicId"
GROUP BY "Comic"."Title"
ORDER BY "TotalSold" DESC
LIMIT 5


In [28]:
db.run(response)

"[('Wolverine: Old Man Logan', 3), ('V for Vendetta', 2), ('Usagi Yojimbo Volume 1', 2), ('Transformers: All Hail Megatron', 2), ('The Killing Joke', 2)]"

In [29]:
# execulatble workflow chian

In [30]:
from langchain_community.tools import QuerySQLDatabaseTool

In [31]:
execute_query_tool = QuerySQLDatabaseTool(db=db)
execute_query_tool

QuerySQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x106f49370>)

In [32]:
query_write_chain = create_sql_query_chain(
    llm=llm_gpt,
    db=db,
    prompt=SQLITE_PROMPT,
    k=10
)

query_execute_chain = (
    query_write_chain
    |
    execute_query_tool
)


In [33]:
query_execute_chain.invoke({"question": "Top 5 most popular comics"})

"[('Wolverine: Old Man Logan', 3), ('V for Vendetta', 2), ('Usagi Yojimbo Volume 1', 2), ('Transformers: All Hail Megatron', 2), ('The Killing Joke', 2)]"

In [34]:
query_execute_chain.invoke({"question": "Top 5 customers with most comics purchased"})

"[('Tony', 'Stark', 8), ('Sarah', 'Connor', 7), ('Natasha', 'Romanoff', 6), ('Clark', 'Kent', 6), ('Diana', 'Prince', 6)]"

In [36]:
query_execute_chain.invoke({"question": "Top 5 customers with most money spent"})

"[('Tony', 'Stark', 164.94), ('Bruce', 'Wayne', 139.94), ('Sarah', 'Connor', 124.96), ('Clark', 'Kent', 114.96), ('Natasha', 'Romanoff', 111.96000000000001)]"

In [37]:
# Text2SQL AI workflow

In [44]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [46]:
answer_prompt = PromptTemplate.from_template("""
    Given the following user question, corresponding SQL query, and SQL results,
    create a helpful answer the user question.

    When generating the final answer in the markdown from the results,
    if there are special characters in the text such as the dollar symbol,
    ensure they are escaped properly for the correct rendering e.g $25.5 should become \$25.5

    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer:
    
""")


text2sql_chain = (
    RunnablePassthrough.assign(query=query_write_chain)
    |
    RunnablePassthrough.assign(result=query_execute_chain)
    |
    answer_prompt
    |
    llm_gpt
    |
    StrOutputParser()
    
)

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/var/folders/v6/yl2g8s4j2837v29lvdsbtzwm0000gp/T/ipykernel_78773/3250221079.py:1: SyntaxWarning: invalid escape sequence '\$'
  answer_prompt = PromptTemplate.from_template("""


In [47]:
from IPython.display import display, Markdown

In [48]:
response = text2sql_chain.invoke({"question":"Total number of customers"})
display(Markdown(response))

The total number of customers is **20**.

In [49]:
response = text2sql_chain.invoke({"question":"what are the Top 10 most popular comics"})
display(Markdown(response))

Here are the Top 10 most popular comics based on sales:

1. **Wolverine: Old Man Logan** - Sold 3 copies
2. **V for Vendetta** - Sold 2 copies
3. **Usagi Yojimbo Volume 1** - Sold 2 copies
4. **Transformers: All Hail Megatron** - Sold 2 copies
5. **The Killing Joke** - Sold 2 copies
6. **The Boys Volume 1** - Sold 2 copies
7. **Superman: Red Son** - Sold 2 copies
8. **Punisher: Welcome Back, Frank** - Sold 2 copies
9. **Preacher Volume 1** - Sold 2 copies
10. **Ms. Marvel Volume 1** - Sold 2 copies

These comics have been the most popular based on the number of copies sold.